# فصل ۱۳: پردازش داده‌ها در LangChain (Data Processing)
## بخش ۳: کاوش عمیق در متادیتا و فیلترگذاری اسناد (Metadata & Filtering)

### 📌 متادیتا (Metadata) چیست و چرا قلب تپنده RAG مدرن است؟
در درسنامه‌های پیشین با شیء پایه‌ای `Document` آشنا شدیم و دیدیم که هر سند از دو بخش تشکیل شده است:
$$\text{Document} = \text{page\_content} \text{ (متن خام)} + \text{metadata} \text{ (شناسنامه و کانتکست)}$$

اگر `page_content` گوشت و پوست سند باشد، **`metadata` روح و شناسنامه آن است**.
در سامانه‌های هوش مصنوعی تجاری، صرفاً جستجوی معنایی (Semantic Search) بر روی متن کافی نیست؛ زیرا یک مدل زبانی بدون اطلاع از **زمان، منبع، نویسنده و سطح دسترسی** دچار «توهم زمانی» یا «نشت اطلاعات محرمانه» می‌شود.

### 🏷️ دو سطح متادیتا در پردازش اسناد:
1. **در سطح سند (Document Level):**
   - اطلاعات کلی سند را جهت شناسایی فراهم می‌کند: نام فایل (`source`)، آدرس اینترنتی (`url`)، فرمت (`type`)، تاریخ ایجاد یا آخرین تغییر (`last_modified`)، نویسنده (`author`) و زبان (`language`).
2. **در سطح المان (Element Level):**
   - بر اساس ساختار درونی سند در هنگام پیش‌پردازش استخراج می‌شود: نوع المان (پاراگراف، تیتر، جدول، تصویر، زیرنویس)، شماره صفحه (`page`)، و ارتباط والد-فرزندی المان‌ها.

### 🏆 مطالعه موردی: مسابقات علیرضا فیروزجا (فاجعه عدم توجه به متادیتای تاریخ)

فرض کنید ربات پشتیبان ورزشی ما می‌خواهد به سوال کاربر پاسخ دهد:
> **سوال کاربر:** *«آخرین نتیجه مسابقات علیرضا فیروزجا در Speed Chess Championship 2024 چه شد؟»*

در دیتابیس ما دو خبر وجود دارد:
- **خبر ۱ (۱۰ مرداد ۱۴۰۳ / ۳۱ ژوئیه ۲۰۲۴):** *«علیرضا فیروزجا قرار است فردا در مرحله یک‌هشتم نهایی به مصاف حریف رود. کارشناسان پیش‌بینی رقابتی نفس‌گیر دارند.»*
- **خبر ۲ (۱۷ مرداد ۱۴۰۳ / ۷ اوت ۲۰۲۴):** *«علیرضا فیروزجا با شکست مقتدرانه حریف خود، به مرحله نیمه‌نهایی مسابقات شطرنج سرعت ۲۰۲۴ صعود کرد.»*

اگر فیلتر متادیتای تاریخ (`date`) وجود نداشته باشد، جستجوی برداری ممکن است به دلیل تشابه کلمات، خبر ۱۰ مرداد را بازیابی کند و مدل به اشتباه بگوید: *«مسابقه هنوز برگزار نشده است!»*
بیایید ابتدا این سناریو را شبیه‌سازی کنیم و اهمیت متادیتا را در عمل ببینیم:

In [1]:
from langchain_core.documents import Document

# ساخت دو سند خبری فیروزجا با متادیتای دقیق تاریخ
news_old = Document(
    page_content="علیرضا فیروزجا در مرحله یک‌هشتم نهایی مسابقات شطرنج سرعتی ۲۰۲۴ آماده مسابقه است و هنوز نتیجه بازی مشخص نیست.",
    metadata={"source": "ورزش۳", "date": "2024-07-31", "topic": "chess", "player": "Alireza Firouzja"}
)

news_recent = Document(
    page_content="علیرضا فیروزجا با پیروزی درخشان در مسابقه، حریف را شکست داد و رسماً به نیمه‌نهایی صعود کرد.",
    metadata={"source": "ورزش۳", "date": "2024-08-07", "topic": "chess", "player": "Alireza Firouzja"}
)

chess_docs = [news_old, news_recent]

# فیلترگذاری هوشمند بر اساس متادیتای تاریخ (انتخاب جدیدترین خبر)
latest_news = max(chess_docs, key=lambda d: d.metadata["date"])

print("📰 اسناد موجود:")
for d in chess_docs:
    print(f"- تاریخ {d.metadata['date']}: {d.page_content}")

print(f"\n✅ جدیدترین خبر انتخاب‌شده با فیلتر متادیتا:\n{latest_news.page_content}")

📰 اسناد موجود:
- تاریخ 2024-07-31: علیرضا فیروزجا در مرحله یک‌هشتم نهایی مسابقات شطرنج سرعتی ۲۰۲۴ آماده مسابقه است و هنوز نتیجه بازی مشخص نیست.
- تاریخ 2024-08-07: علیرضا فیروزجا با پیروزی درخشان در مسابقه، حریف را شکست داد و رسماً به نیمه‌نهایی صعود کرد.

✅ جدیدترین خبر انتخاب‌شده با فیلتر متادیتا:
علیرضا فیروزجا با پیروزی درخشان در مسابقه، حریف را شکست داد و رسماً به نیمه‌نهایی صعود کرد.


### ۱. تابع کمکی چاپ متادیتای اسناد

طبق درسنامه، تابع کمکی `print_metadata` را تعریف می‌کنیم تا متادیتای تمام اسناد موجود در یک لیست را در خروجی نمایش دهد:

In [2]:
def print_metadata(list_of_docs):
    """چاپ متادیتای تمام اسناد موجود در یک لیست"""
    for doc in list_of_docs:
        print(doc.metadata)

print("تابع print_metadata با موفقیت تعریف شد.")

تابع print_metadata با موفقیت تعریف شد.


### ۲. بارگیری سند متنی فارسی (`TextLoader`)

ابتدا سخنرانی معروف مارتین لوتر کینگ را به زبان فارسی از فایل `I_have_a_dream_fa.txt` بارگیری می‌کنیم.
به طور پیش‌فرض، لودر `TextLoader` فقط متادیتای `source` (نام یا مسیر فایل) را به سند الصاق می‌کند:

In [3]:
from langchain_community.document_loaders import TextLoader

txt_path = "I_have_a_dream_fa.txt"
txt_loader = TextLoader(txt_path, encoding="utf-8")
txt_file = txt_loader.load()

print("متادیتای پیش‌فرض لودر متنی:")
print_metadata(txt_file)

C:\Users\aline\AppData\Local\Temp\ipykernel_19972\329111411.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


متادیتای پیش‌فرض لودر متنی:
{'source': 'I_have_a_dream_fa.txt'}


### ۳. بارگیری سند PDF انگلیسی (`PyPDFium2Loader` / `PyPDFLoader`)

سپس فایل نسخه انگلیسی را از فایل `I_have_a_dream_en.pdf` بارگیری می‌کنیم.
لودرهای PDF علاوه بر `source`، کلید ارزشمند **`page`** (شماره صفحه که از صفر شروع می‌شود) را نیز در سطح المان به متادیتا اضافه می‌کنند:

In [4]:
import importlib.util

# استفاده از PyPDFium2Loader در صورت نصب بودن یا fallback به PyPDFLoader
if importlib.util.find_spec("pypdfium2"):
    from langchain_community.document_loaders import PyPDFium2Loader as PDFLoader
    pdf_engine = "PyPDFium2Loader"
else:
    from langchain_community.document_loaders import PyPDFLoader as PDFLoader
    pdf_engine = "PyPDFLoader (جایگزین استاندارد)"

file_path_pdf = "I_have_a_dream_en.pdf"
pdfloader = PDFLoader(file_path_pdf)
pdf_file = pdfloader.load()

# برای یکنواختی کامل کلیدهای متادیتا مشابه درسنامه کوئرا
for doc in pdf_file:
    doc.metadata = {"source": file_path_pdf, "page": doc.metadata.get("page", 0)}

print(f"لودر استفاده‌شده: {pdf_engine}")
print("متادیتای صفحات استخراج‌شده از PDF:")
print_metadata(pdf_file)

لودر استفاده‌شده: PyPDFLoader (جایگزین استاندارد)
متادیتای صفحات استخراج‌شده از PDF:
{'source': 'I_have_a_dream_en.pdf', 'page': 0}
{'source': 'I_have_a_dream_en.pdf', 'page': 1}
{'source': 'I_have_a_dream_en.pdf', 'page': 2}


### ۴. غنی‌سازی دستی متادیتا (Metadata Enrichment)

متادیتای خام تولیدشده توسط لودرها بسیار محدود است. در عمل ما می‌خواهیم فیلدهای تجاری مورد نظرمان را به هر سند اضافه کنیم:
- `type`: نوع فایل (`text` یا `pdf`)
- `author`: نویسنده اثر (`Martin Luther King` یا `Martin Luther King Jr.`)
- `language`: زبان محتوا (`fa` یا `en`)

دیکشنری `doc.metadata` یک ساختار استاندارد پایتونی است و به راحتی با کلیدگذاری مستقیم ویرایش می‌شود:

In [5]:
# افزودن متادیتای تکمیلی به اسناد فایل متنی فارسی
for doc in txt_file:
    doc.metadata["type"] = "text"
    doc.metadata["author"] = "Martin Luther King"
    doc.metadata["language"] = "fa"

print("متادیتای اسناد متنی فارسی پس از غنی‌سازی:")
print_metadata(txt_file)
print("-" * 60)

# افزودن متادیتای تکمیلی به اسناد فایل PDF انگلیسی
for doc in pdf_file:
    doc.metadata["type"] = "pdf"
    doc.metadata["author"] = "Martin Luther King Jr."
    doc.metadata["language"] = "en"

print("متادیتای اسناد PDF انگلیسی پس از غنی‌سازی:")
print_metadata(pdf_file)

متادیتای اسناد متنی فارسی پس از غنی‌سازی:
{'source': 'I_have_a_dream_fa.txt', 'type': 'text', 'author': 'Martin Luther King', 'language': 'fa'}
------------------------------------------------------------
متادیتای اسناد PDF انگلیسی پس از غنی‌سازی:
{'source': 'I_have_a_dream_en.pdf', 'page': 0, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 1, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 2, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}


### ۵. ادغام اسناد در یک مجموعه واحد (`all_docs`)

اکنون تمام اسناد را در یک لیست ترکیب می‌کنیم تا کالکشن نهایی اسناد سامانه را تشکیل دهیم:

In [6]:
# ترکیب هر دو لیست اسناد در یک کالکشن واحد
all_docs = txt_file + pdf_file

print(f"تعداد کل اسناد: {len(all_docs)}\n")
print_metadata(all_docs)

تعداد کل اسناد: 4

{'source': 'I_have_a_dream_fa.txt', 'type': 'text', 'author': 'Martin Luther King', 'language': 'fa'}
{'source': 'I_have_a_dream_en.pdf', 'page': 0, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 1, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 2, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}


### ۶. فیلترگذاری اسناد بر اساس متادیتا (Filtering)

اکنون به قدرت واقعی متادیتا می‌رسیم! فرض کنید می‌خواهیم بخش‌های خاصی از داده را برای کاربردهای مجزا جدا کنیم:

In [7]:
# فیلتر ۱: استخراج اسناد زبان فارسی با متادیتای language
persian_docs = [doc for doc in all_docs if doc.metadata.get("language") == "fa"]
print("🇮🇷 اسناد فارسی:")
print_metadata(persian_docs)
print("=" * 60)

# فیلتر ۲: استخراج اسناد برگرفته از PDF با متادیتای type
pdf_docs = [doc for doc in all_docs if doc.metadata.get("type") == "pdf"]
print("📄 اسناد حاصل از فایل PDF:")
print_metadata(pdf_docs)
print("=" * 60)

# فیلتر ۳: فیلتر ترکیبی (اسناد PDF انگلیسی با شماره صفحه زوج)
even_page_docs = [
    doc for doc in all_docs 
    if doc.metadata.get("type") == "pdf" and doc.metadata.get("page", 1) % 2 == 0
]
print("🎯 اسناد PDF انگلیسی در صفحات زوج (0, 2):")
print_metadata(even_page_docs)

🇮🇷 اسناد فارسی:
{'source': 'I_have_a_dream_fa.txt', 'type': 'text', 'author': 'Martin Luther King', 'language': 'fa'}
📄 اسناد حاصل از فایل PDF:
{'source': 'I_have_a_dream_en.pdf', 'page': 0, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 1, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 2, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
🎯 اسناد PDF انگلیسی در صفحات زوج (0, 2):
{'source': 'I_have_a_dream_en.pdf', 'page': 0, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}
{'source': 'I_have_a_dream_en.pdf', 'page': 2, 'type': 'pdf', 'author': 'Martin Luther King Jr.', 'language': 'en'}


### ۷. استفاده از متادیتا در پایگاه داده برداری (Vector DB Integration)

در سیستم‌های RAG واقعی، اسناد را در پایگاه برداری (مانند Chroma) ذخیره کرده و فیلتر متادیتا را مستقیماً در هنگام کوئری اعمال می‌کنیم (Pre-filtering):
این کار باعث می‌شود جستجوی برداری فقط در بین اسناد مجاز انجام شود:

In [8]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# مدل امبدینگ چندزبانه
emb_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# بارگذاری تمام اسناد غنی‌شده در Chroma
vector_store = Chroma.from_documents(
    documents=all_docs,
    embedding=emb_model
)

# کوئری جستجو فقط بر روی اسناد فارسی (با فیلتر متادیتا)
query = "رویای آزادی و برابری انسان‌ها"
results = vector_store.similarity_search(
    query,
    k=2,
    filter={"language": "fa"}  # اعمال فیلتر متادیتا در سطح دیتابیس
)

print(f"🔍 کوئری: '{query}' با فیلتر {{'language': 'fa'}}\n")
for r in results:
    print(f"متادیتا: {r.metadata}")
    print(f"متن بازیابی‌شده: {r.page_content[:150]}...\n")

d:\Project\LLM\llm-apps-course\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not reach the Hub (Cannot reach https://huggingface.co/api/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2: offline mode is enabled. To disable it, please unset the `HF_HUB_OFFLINE` environment variable.). Using cached commit hash for 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1991.81it/s]


🔍 کوئری: 'رویای آزادی و برابری انسان‌ها' با فیلتر {'language': 'fa'}

متادیتا: {'language': 'fa', 'author': 'Martin Luther King', 'source': 'I_have_a_dream_fa.txt', 'type': 'text'}
متن بازیابی‌شده: از اینکه امروز در تظاهراتی کنار شما هستم که روزی در تاریخ ملت ما به عنوان بزرگترین رخداد در راستای آزادی به ثبت خواهد رسید، خوشحالم.
یکصد سال پیش، یک ...



### 🎯 جمع‌بندی درسنامه متادیتا:

1. **تعریف:** متادیتا بافتار، هویت و اطلاعات ساختاری سند را فراهم می‌کند.
2. **دو سطح اصلی:** سطح سند (شناسه، تاریخ، منبع) و سطح المان (نوع قطعه، شماره صفحه، جایگاه ساختاری).
3. **ارزش در RAG:** جلوگیری از خطای توهم زمانی (مثل مثال شطرنج‌باز فیروزجا) و ایجاد سیستم‌های جستجوی هیبریدی.
4. **دسترسی و دستکاری:** دیکشنری `doc.metadata` به سادگی قابل خواندن و غنی‌سازی با فیلدهای دلخواه است.
5. **فیلترگذاری در پایگاه داده برداری:** با استفاده از آرگومان `filter` در متد `similarity_search`، اسناد نامربوط قبل از مقایسه برداری حذف می‌شوند.